# L2: DSPy Programming - Signatures and Modules

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [1]:
from helper import get_openai_api_key
openai_api_key = get_openai_api_key()

import os

os.environ["OPENAI_API_KEY"]  = get_openai_api_key()

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>
</div>

### Set up API key

In [2]:
from helper import get_openai_api_key
openai_api_key = get_openai_api_key()

import os

os.environ["OPENAI_API_KEY"] = get_openai_api_key()

### Configure the LM

In [3]:
import dspy
dspy.settings.configure(lm=dspy.LM("openai/gpt-4o-mini"))

## Use DSPy built-in Module to Build a Sentiment Classifier

In [4]:
class SentimentClassifier(dspy.Signature):
    """Classify the sentiment of a text."""

    text: str = dspy.InputField(desc="input text to classify sentiment")
    sentiment: int = dspy.OutputField(
        desc="sentiment, the higher the more positive", ge=0, le=10
    )

In [5]:
str_signature = dspy.make_signature("text -> sentiment")

### Create a Module to Interact with the LM

In [6]:
predict = dspy.Predict(SentimentClassifier) 

In [7]:
output = predict(text="I am feeling pretty happy!")
print(output)

Prediction(
    sentiment=8
)


In [8]:
print(f"The sentiment is: {output.sentiment}")
print(f"The sentiment is: {output['sentiment']}")

The sentiment is: 8
The sentiment is: 8


In [9]:
dspy.configure(lm=dspy.LM("openai/gpt-4o"))
print(predict(text="I am feeling pretty happy!"))

Prediction(
    sentiment=8
)


In [10]:
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))

### Wait, Where is My Prompt? 

In [11]:
dspy.inspect_history(n=1)





[2025-06-07T03:00:44.376474]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling pretty happy!

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
8

[[ ## completed ## ]]







### Try a Different Built-in Module

In [12]:
cot = dspy.ChainOfThought(SentimentClassifier)

output = cot(text="I am feeling pretty happy!")
print(output)

Prediction(
    reasoning='The text expresses a positive emotion, specifically happiness. The use of the word "happy" indicates a strong positive sentiment. Therefore, the sentiment score is high.',
    sentiment=8
)


In [13]:
dspy.inspect_history(n=1)





[2025-06-07T03:00:44.410601]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `reasoning` (str)
2. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling pretty happy!

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## sentiment ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
The 

### Use a Different Adapter

In [14]:
dspy.configure(adapter=dspy.JSONAdapter())

In [15]:
print(cot(text="I am feeling pretty happy!"))
dspy.inspect_history(n=1)

Prediction(
    reasoning="The text expresses a positive emotion, specifically happiness, which indicates a favorable sentiment. The use of the word 'happy' suggests a strong positive feeling.",
    sentiment=8
)




[2025-06-07T03:00:44.450634]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `reasoning` (str)
2. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "sentiment": "{sentiment}        # note: the value you produce must be a single int value"
}
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeli

## Build a Program with Custom Module

In [16]:
class QuestionGenerator(dspy.Signature):
    """Generate a yes or no question in order to guess the celebrity name in users' mind. You can ask in general or directly guess the name if you think the signal is enough. You should never ask the same question in the past_questions."""
    past_questions: list[str] = dspy.InputField(desc="past questions asked")
    past_answers: list[bool] = dspy.InputField(desc="past answers")
    new_question: str = dspy.OutputField(desc="new question that can help narrow down the celebrity name")
    guess_made: bool = dspy.OutputField(desc="If the new_question is the celebrity name guess, set to True, if it is still a general question set to False")


class Reflection(dspy.Signature):
    """Provide reflection on the guessing process"""
    correct_celebrity_name: str = dspy.InputField(desc="the celebrity name in user's mind")
    final_guessor_question: str = dspy.InputField(desc="the final guess or question LM made")
    past_questions: list[str] = dspy.InputField(desc="past questions asked")
    past_answers: list[bool] = dspy.InputField(desc="past answers")

    reflection: str = dspy.OutputField(
        desc="reflection on the guessing process, including what was done well and what can be improved"
    )

def ask(prompt, valid_responses=("y", "n")):
    while True:
        response = input(f"{prompt} ({'/'.join(valid_responses)}): ").strip().lower()
        if response in valid_responses:
            return response
        print(f"Please enter one of: {', '.join(valid_responses)}")

class CelebrityGuess(dspy.Module):
    def __init__(self, max_tries=10):
        super().__init__()

        self.question_generator = dspy.ChainOfThought(QuestionGenerator)
        self.reflection = dspy.ChainOfThought(Reflection)

        self.max_tries = 20

    def forward(self):
        celebrity_name = input("Please think of a celebrity name, once you are ready, type the name and press enter...")
        past_questions = []
        past_answers = []

        correct_guess = False

        for i in range(self.max_tries):
            question = self.question_generator(
                past_questions=past_questions,
                past_answers=past_answers,
            )
            answer = ask(f"{question.new_question}").lower() == "y"
            past_questions.append(question.new_question)
            past_answers.append(answer)

            if question.guess_made and answer:
                correct_guess = True
                break

        if correct_guess:
            print("Yay! I got it right!")
        else:
            print("Oops, I couldn't guess it right.")

        reflection = self.reflection(
            correct_celebrity_name=celebrity_name,
            final_guessor_question=question.new_question,
            past_questions=past_questions,
            past_answers=past_answers,
        )
        print(reflection.reflection)


In [17]:
celebrity_guess = CelebrityGuess()

In [18]:
celebrity_guess()

Please think of a celebrity name, once you are ready, type the name and press enter...sylvestre stalone
Is the celebrity an actor or actress? (y/n): actor
Please enter one of: y, n
Is the celebrity an actor or actress? (y/n): y
Is the celebrity primarily known for their work in film? (y/n): y
Is the celebrity known for their work in action films? (y/n): y
Is the celebrity known for their role in a specific action film franchise, like 'Mission: Impossible' or 'Fast & Furious'? (y/n): y
Is the celebrity known for their role in 'The Avengers' franchise? (y/n): n
Is the celebrity known for their role in the 'Transformers' franchise? (y/n): n
Is the celebrity known for their role in the 'James Bond' franchise? (y/n): n
Is the celebrity known for their role in the 'Mission: Impossible' franchise? (y/n): n
Is the celebrity known for their role in the 'Fast & Furious' franchise? (y/n): n
Is the celebrity Tom Cruise? (y/n): n
Is the celebrity known for their role in the 'John Wick' franchise? (

## Save and Load

In [19]:
celebrity_guess.save("dspy_program/celebrity.json", save_program=False)

In [20]:
celebrity_guess.load("dspy_program/celebrity.json")

In [21]:
celebrity_guess.save("dspy_program/celebrity/", save_program=True)

In [22]:
loaded = dspy.load("dspy_program/celebrity/")

In [23]:
loaded()

Please think of a celebrity name, once you are ready, type the name and press enter...sylvestre stalone
Is the celebrity an actor or actress? (y/n): y
Is the celebrity primarily known for their work in film? (y/n): y
Is the celebrity known for their work in action films? (y/n): y
Is the celebrity known for their role in a specific action film franchise, like 'Mission: Impossible' or 'Fast & Furious'? (y/n): y
Is the celebrity known for their role in 'The Avengers' franchise? (y/n): n
Is the celebrity known for their role in the 'Transformers' franchise? (y/n): n
Is the celebrity known for their role in the 'James Bond' franchise? (y/n): n
Is the celebrity known for their role in the 'Mission: Impossible' franchise? (y/n): n
Is the celebrity known for their role in the 'Fast & Furious' franchise? (y/n): n
Is the celebrity Tom Cruise? (y/n): n
Is the celebrity known for their role in the 'John Wick' franchise? (y/n): n
Is the celebrity Keanu Reeves? (y/n): n
Is the celebrity known for th